In [0]:
%run ../00_common/data_utils

#config 

#config fun

#Ingest

#Merge

#ukey group

#Cid group

In [0]:
## ===================== 环境配置 =======================
# Kafka broker 地址
kafka_brokers = get_env_config("target_kafka.kafka_brokers")
print(f"kafka_brokers: {kafka_brokers}")
# database & table 配置
golden_touchpoint_combine_database = get_env_config("golden_touchpoint_combine_database")
print(f"golden_touchpoint_combine_database: {golden_touchpoint_combine_database}")
## ======================================================

apac_touchpoint_topic_name = "TouchPointMaster"
kor_touchpoint_topic_name = "TouchPointMaster_KR"

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
def generate_touchpoint_json_data(condition_str):
    touchpoint_df = (spark.table(f"{golden_touchpoint_combine_database}.t_touchpoint_dataset")
        .where(condition_str)
        .select(
            F.concat_ws("_", F.col("MarketCode"), F.col("BrandCode"), F.col("TouchPointCode")).alias("key"),
            F.col("TouchPointMasterJSON").alias("value"),
            F.when(F.col("MarketCode") == "KOR", kor_touchpoint_topic_name).otherwise(apac_touchpoint_topic_name).alias("topic")
        )
    )
    
    return touchpoint_df

def send_touchpoint_topic(touchpoint_json_df):
    (
        touchpoint_json_df
            .write.format("kafka")
            .option("kafka.bootstrap.servers", kafka_brokers)
            .option("kafka.request.timeout.ms", "15000")
            .option("kafka.max.block.ms", "20000")
            .option("kafka.delivery.timeout.ms", "30000")
            .option("kafka.retries", "0")
            .mode("append")
            .save()
    )


def touchpoin_retry_downstream(kafka_brokers, golden_touchpoint_combine_database, condition_str):
    touchpoint_df = generate_touchpoint_json_data(condition_str)
    
    print(f"send count by topic")
    display(touchpoint_df.groupBy("topic").count())

    send_touchpoint_topic(touchpoint_df)
    print(f"send touchpoint topic success: {kafka_brokers}")

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
'''
Example of Condition_str:
    (MarketCode = 'AUS' and BrandCode = '01' and TouchPointCode = '123456') or
    (MarketCode = 'HKG' and BrandCode = '03' and TouchPointCode = 'HKCL0052') or
    (MarketCode = 'KOR' and BrandCode = '01' and TouchPointCode = '012') or
    (MarketCode = 'KOR' and BrandCode = '01' and TouchPointCode = '013')

'''

dbutils.widgets.text("condition_str", "1=0")
condition_str = dbutils.widgets.get("condition_str")
print(f"condition_str: {condition_str}")

touchpoin_retry_downstream(kafka_brokers, golden_touchpoint_combine_database, condition_str)

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage